In [ ]:
# ============================================================
# PHASE 4 DEEP DIVE: Proper time-series cross-validation
# ============================================================
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, roc_auc_score
import xgboost as xgb

# Prepare the final dataset
df_model = df_features[feature_cols + ['Target']].dropna()

X = df_model[feature_cols]
y = df_model['Target']

print(f"Dataset: {len(X)} samples, {len(feature_cols)} features")
print(f"Target balance: {y.mean():.3%} positive (up days)")

In [ ]:
# --- Walk-forward (time-series) cross-validation ---
# Unlike random k-fold CV, TimeSeriesSplit always trains on the past
# and tests on the future. This mirrors real deployment conditions.
#
#  Fold 1: |--train--|--test--|
#  Fold 2: |----train----|--test--|
#  Fold 3: |------train------|--test--|
#  ... etc.
#
# This is the only correct way to cross-validate time series data.

tscv = TimeSeriesSplit(n_splits=5, gap=1)
# gap=1 skips one row between train and test — prevents leakage at boundaries

# Build model pipelines (scaler + model in one object)
# Pipelines are critical: they ensure the scaler is fit only on training data
pipelines = {
    "LogisticReg": Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(C=0.1, max_iter=1000, random_state=42))
    ]),
    "RandomForest": Pipeline([
        ('scaler', StandardScaler()),
        ('model', RandomForestClassifier(n_estimators=300, max_depth=6,
                                         min_samples_leaf=20, random_state=42))
    ]),
    "GradientBoosting": Pipeline([
        ('scaler', StandardScaler()),
        ('model', GradientBoostingClassifier(n_estimators=200, max_depth=3,
                                              learning_rate=0.05, subsample=0.8,
                                              random_state=42))
    ]),
    "XGBoost": Pipeline([
        ('scaler', StandardScaler()),
        ('model', xgb.XGBClassifier(n_estimators=300, max_depth=4,
                                     learning_rate=0.03, subsample=0.8,
                                     colsample_bytree=0.8, reg_alpha=0.1,
                                     eval_metric='logloss', verbosity=0,
                                     random_state=42))
    ])
}

# Run walk-forward CV for all models
cv_results = {}
for name, pipeline in pipelines.items():
    acc_scores = cross_val_score(pipeline, X, y, cv=tscv, scoring='accuracy')
    auc_scores = cross_val_score(pipeline, X, y, cv=tscv, scoring='roc_auc')
    cv_results[name] = {
        'acc_mean':  acc_scores.mean(),
        'acc_std':   acc_scores.std(),
        'auc_mean':  auc_scores.mean(),
        'auc_std':   auc_scores.std(),
        'acc_scores': acc_scores
    }
    print(f"\n{name}:")
    print(f"  Accuracy: {acc_scores.mean():.4f} ± {acc_scores.std():.4f}")
    print(f"  AUC-ROC:  {auc_scores.mean():.4f} ± {auc_scores.std():.4f}")
    print(f"  Per-fold: {[f'{s:.3f}' for s in acc_scores]}")

In [ ]:
# --- Hyperparameter tuning with TimeSeriesSplit ---
# We tune the XGBoost model using RandomizedSearchCV (faster than GridSearch)
from sklearn.model_selection import RandomizedSearchCV

xgb_param_grid = {
    'model__n_estimators':     [100, 200, 300, 500],
    'model__max_depth':        [2, 3, 4, 5],
    'model__learning_rate':    [0.01, 0.03, 0.05, 0.1],
    'model__subsample':        [0.6, 0.7, 0.8, 0.9],
    'model__colsample_bytree': [0.6, 0.7, 0.8, 0.9],
    'model__reg_alpha':        [0, 0.01, 0.1, 1.0],
    'model__reg_lambda':       [1.0, 2.0, 5.0],
}

xgb_search = RandomizedSearchCV(
    pipelines["XGBoost"],
    param_distributions=xgb_param_grid,
    n_iter=30,                  # Try 30 random combinations (not all ~5000)
    cv=tscv,
    scoring='roc_auc',
    n_jobs=-1,                  # Use all CPU cores
    random_state=42,
    verbose=1
)

# NOTE: This will take 2-4 minutes to run
print("Tuning XGBoost hyperparameters...")
xgb_search.fit(X, y)
print(f"\nBest AUC: {xgb_search.best_score_:.4f}")
print(f"Best params: {xgb_search.best_params_}")

best_xgb = xgb_search.best_estimator_

In [ ]:
# --- Final holdout evaluation ---
# We use the LAST 15% of data as a completely untouched holdout set.
# This simulates real deployment: you train on historical data, test on future.

holdout_split = int(len(X) * 0.85)
X_train_full = X.iloc[:holdout_split]
X_holdout    = X.iloc[holdout_split:]
y_train_full = y.iloc[:holdout_split]
y_holdout    = y.iloc[holdout_split:]

print(f"\nFinal evaluation on holdout set ({len(X_holdout)} days):")
print("=" * 55)

final_results = {}
for name, pipeline in pipelines.items():
    pipeline.fit(X_train_full, y_train_full)
    preds = pipeline.predict(X_holdout)
    proba = pipeline.predict_proba(X_holdout)[:, 1]

    acc = accuracy_score(y_holdout, preds)
    auc = roc_auc_score(y_holdout, proba)
    final_results[name] = {'accuracy': acc, 'auc': auc,
                            'proba': proba, 'preds': preds}
    print(f"{name:20s}  Accuracy: {acc:.4f}  AUC: {auc:.4f}")

# Also evaluate best tuned XGBoost
best_xgb.fit(X_train_full, y_train_full)
tuned_preds = best_xgb.predict(X_holdout)
tuned_proba = best_xgb.predict_proba(X_holdout)[:, 1]
print(f"{'XGBoost (tuned)':20s}  Accuracy: {accuracy_score(y_holdout, tuned_preds):.4f}  "
      f"AUC: {roc_auc_score(y_holdout, tuned_proba):.4f}")